# RAGAS-style RAG eval — four metrics, four failure modes

A RAG system can fail in five distinct ways. Recall@k catches one. The other four are caught by these LLM-graded metrics:

| Metric | What it checks | Failure caught |
|---|---|---|
| **Context Precision** | Of retrieved chunks, how many are useful? | Retriever returns junk |
| **Context Recall** | Of reference claims, how many are recoverable from retrieved chunks? | Retriever misses key docs |
| **Faithfulness** | Every answer claim supported by context? | Hallucination |
| **Answer Relevancy** | Does the answer actually address the question? | Off-topic / evasive |

We re-implement them on top of `shared.llm.complete()` so the prompts are visible and the cache works.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


In [ ]:
import sys, pathlib
leaf = pathlib.Path.cwd()
sys.path.insert(0, str(leaf))
from eval import (
    answer_relevancy, context_precision, context_recall, faithfulness,
)
from shared.embedders import cosine_topk, hash_embed
from shared.llm import Message, complete
from shared.loaders import load_corpus, load_golden_qa
from shared.prompts import RAG_SYSTEM, rag_user_prompt

DOCS = load_corpus()
QA = [q for q in load_golden_qa() if q.source_ids]
DIMS = 256
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids = [d.arxiv_id for d in DOCS]
doc_vecs = hash_embed(doc_texts, dims=DIMS, seed=0)
print(f'{len(DOCS)} docs, {len(QA)} questions')

## Pull a single example through the full pipeline

In [ ]:
def retrieve(question, k=3):
    qv = hash_embed([question], dims=DIMS, seed=0)[0]
    idx, _ = cosine_topk(qv, doc_vecs, k=k)
    return [(doc_ids[i], doc_texts[i]) for i in idx]

def generate(question, retrieved):
    try:
        return complete(
            model='openai/gpt-4o-mini',
            namespace='05-evals-and-observability/00-ragas-rag-eval',
            messages=[
                Message(role='system', content=RAG_SYSTEM),
                Message(role='user', content=rag_user_prompt(question, retrieved)),
            ],
        ).content
    except Exception:
        return retrieved[0][1] if retrieved else ''

demo = QA[1]  # q02 — direct, answerable
retrieved = retrieve(demo.question)
answer = generate(demo.question, retrieved)
ctxs = [t for _, t in retrieved]
print('Q:', demo.question)
print('A:', answer[:200])
print()
print('context_precision :', context_precision(demo.question, ctxs))
print('context_recall    :', context_recall(demo.question, demo.answer, ctxs))
print('faithfulness      :', faithfulness(answer, ctxs))
print('answer_relevancy  :', answer_relevancy(demo.question, answer))

## Interpreting the metrics

* **Low context_precision** → tune the retriever (better embedder, hybrid search, reranker).
* **Low context_recall** → increase `k`, expand queries (HyDE / multi-query), or chunk smaller.
* **Low faithfulness** → tighten the system prompt ("answer ONLY from the provided context"), use a stronger model, or add Self-RAG-style critique.
* **Low answer_relevancy** → the model is hedging or going off-topic; reduce temperature, sharpen the system prompt, or post-filter.

The four metrics are mostly orthogonal — you can have high faithfulness with low recall (model correctly says 'I don't know') or high recall with low faithfulness (model embellishes). Track all four.

## What you can extend

* Add **answer correctness** as a fifth metric — semantic-similarity to a reference answer (RAGAS calls this `answer_similarity` + `answer_correctness`).
* Use **paired-bootstrap** to compare two RAG configs with p-values, not just deltas.
* Swap the judge model and recompute — same prompts, different scores. Track judge-model bias.
* Compare to `ragas` library directly (`uv add ragas` + run the same QA set).